# Single Session Pre-processing Workflow
Set `mouse` and `session` below, then run the processing cell to reproduce the pipeline from `preprocess_ephys_behavior_data.py` without interactive prompts.

In [ ]:
from pathlib import Path
import traceback
from typing import Any, Dict, Optional

import matplotlib.pyplot as plt
import numpy as np
import spks  # type: ignore

# # Ensure the repository root (containing the `ephys` package) is on sys.path
# candidate_roots = {
#     Path().resolve(),
#     Path().resolve().parent,
#     Path().resolve().parent.parent,
# }
# for path in candidate_roots:
#     if (path / "ephys").exists() and str(path) not in sys.path:
#         sys.path.insert(0, str(path))
#         break
# else:
#     raise RuntimeError(
#         "Could not locate the repository root containing the `ephys` package."
#     )

from ephys.utils.utils_IO import (
    load_sync_data,
    process_port_events,
    process_trial_data,
)
from ephys.utils.utils_neural import (
    get_cluster_spike_times,
    get_good_units,
)

%matplotlib widget

In [ ]:
# Configure the session you want to process
mouse = "GRB006"
session = "20240821_121447"

data_path = Path("/Volumes/grb_ephys/data")
local_data_path = Path("/Users/gabriel/data")
save_locally = False  # set to False to write alongside raw data
overwrite = True  # set to True to regenerate outputs even if they exist
write_outputs = True  # set to True to persist spike times and trial tables

In [ ]:
def process_session(
    mouse: str,
    session: str,
    *,
    data_path: Path,
    save_locally: bool = False,
    local_data_path: Optional[Path] = None,
    overwrite: bool = False,
    write_outputs: bool = True,
) -> Dict[str, Any]:
    """Run the preprocessing pipeline for a single session."""
    session_path = data_path / mouse / session
    if not session_path.exists():
        raise FileNotFoundError(f"Session path not found: {session_path}")

    if save_locally:
        if local_data_path is None:
            raise ValueError("local_data_path must be provided when save_locally=True.")
        base_path = local_data_path
    else:
        base_path = data_path

    kilosort_path = session_path / "kilosort2.5" / "imec0"
    if not kilosort_path.exists():
        raise FileNotFoundError(f"Kilosort directory not found: {kilosort_path}")

    if write_outputs:
        save_dir = base_path / mouse / session / "pre_processed"
        save_dir.mkdir(parents=True, exist_ok=True)
        spike_times_file = save_dir / "spike_times_per_unit.npy"
        trial_ts_file = save_dir / "trial_ts.pkl"

        if not overwrite and spike_times_file.exists() and trial_ts_file.exists():
            print(f"Skipping {mouse} / {session}: outputs already exist.")
            return {
                "mouse": mouse,
                "session": session,
                "session_path": session_path,
                "save_dir": save_dir,
                "spike_times_file": spike_times_file,
                "trial_ts_file": trial_ts_file,
                "skipped": True,
            }
    else:
        save_dir = None
        spike_times_file = None
        trial_ts_file = None

    print(f"Processing {mouse} / {session}")

    (
        corrected_onsets,
        corrected_offsets,
        t,
        srate,
        analog_signals,
    ) = load_sync_data(str(session_path))
    trial_starts, port_events = process_port_events(
        corrected_onsets, corrected_offsets, srate
    )
    behavior_data, trial_ts, stim_ts_per_channel = process_trial_data(
        str(session_path),
        trial_starts,
        t,
        srate,
        analog_signals,
        port_events,
        mouse,
        session,
    )

    sc = np.load(kilosort_path / "spike_clusters.npy")
    ss = np.load(kilosort_path / "spike_times.npy")
    st = ss / srate

    clu = spks.clusters.Clusters(
        folder=str(kilosort_path),
        get_waveforms=False,
        get_metrics=True,
        load_template_features=True,
    )
    good_units_mask, n_units = get_good_units(clusters_obj=clu, spike_clusters=sc)
    print(f"Good units: {n_units}")

    spike_times_per_unit = get_cluster_spike_times(
        spike_times=st,
        spike_clusters=sc,
        good_unit_ids=good_units_mask,
    )

    result: Dict[str, Any] = {
        "mouse": mouse,
        "session": session,
        "session_path": session_path,
        "save_dir": save_dir,
        "spike_times_file": spike_times_file,
        "trial_ts_file": trial_ts_file,
        "data_path": data_path,
        "kilosort_path": kilosort_path,
        "analog_signals": analog_signals,
        "stim_ts_per_channel": stim_ts_per_channel,
        "corrected_onsets": corrected_onsets,
        "corrected_offsets": corrected_offsets,
        "time_vector": t,
        "sampling_rate": srate,
        "trial_starts": trial_starts,
        "port_events": port_events,
        "behavior_data": behavior_data,
        "trial_ts": trial_ts,
        "spike_times_samples": ss,
        "spike_times_seconds": st,
        "spike_clusters": sc,
        "cluster_object": clu,
        "good_units_mask": good_units_mask,
        "n_units": n_units,
        "spike_times_per_unit": spike_times_per_unit,
        "write_outputs": write_outputs,
        "overwrite": overwrite,
    }

    if write_outputs:
        np.save(
            spike_times_file,
            np.array(spike_times_per_unit, dtype=object),
            allow_pickle=True,
        )
        trial_ts.to_pickle(trial_ts_file)
        print(f"Saved outputs to {save_dir}")
    else:
        print("write_outputs is False; skipping file writes.")

    return result

In [ ]:
session_outputs = {}
try:
    session_outputs = process_session(
        mouse=mouse,
        session=session,
        data_path=data_path,
        save_locally=save_locally,
        local_data_path=local_data_path,
        overwrite=overwrite,
        write_outputs=write_outputs,
    )
except Exception:
    traceback.print_exc()
else:
    print("Processing complete.")
# session_outputs

In [ ]:
time = session_outputs["time_vector"]
visual_signal = session_outputs["analog_signals"]["visual"]
audio_signal = session_outputs["analog_signals"]["audio"]

start_time = 100  # seconds
end_time = 105  # seconds
if start_time < time[0] or end_time > time[-1] or start_time >= end_time:
    raise ValueError(
        "start_time/end_time must be within the recorded window and start < end."
    )

start_idx = int(np.searchsorted(time, start_time))
end_idx = int(np.searchsorted(time, end_time))
end_idx = max(end_idx, start_idx + 1)

fig, ax = plt.subplots(1, figsize=(8, 3))
ax.plot(
    time[start_idx:end_idx], visual_signal[start_idx:end_idx], label="visual", alpha=0.7
)
ax.plot(
    time[start_idx:end_idx], audio_signal[start_idx:end_idx], label="audio", alpha=0.7
)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Signal amplitude (a.u.)")
ax.legend(loc="upper right")
ax.set_title(f"Stimulus channels: {start_time:.3f}s to {end_time:.3f}s")
fig.tight_layout()

In [ ]:
import pickle

# Save session_outputs dictionary to Downloads
downloads_path = Path.home() / "Downloads"
output_file = downloads_path / f"{mouse}_{session}_session_outputs.pkl"

with open(output_file, "wb") as f:
    pickle.dump(session_outputs, f)

print(f"Session outputs saved to: {output_file}")